# Trivialised Diffusion

This notebook is the companion to [`continious.ipynb`](./continious.ipynb), but for the simplified trivialised diffusion on periodic fractional coordinates.

We use the real [`TrivialisedDiffusion`](./trivialized_diffusion.py) class and real crystal batches from the MatterGen-backed dataloaders.

## Theory

The simplified forward process used here is:

1. sample a noisy velocity
   
   $$
   v_t = \alpha_v(t) v_0 + \sigma_v(t) \epsilon_v
   $$

2. use that velocity to define a periodic displacement
   
   $$
   r_t = \mathrm{wrap}(\mu_r(t, v_t, v_0) + \sigma_r(t) \epsilon_r)
   $$

3. move the fractional coordinates on the torus
   
   $$
   f_t = \mathrm{wrap}(f_0 + r_t)
   $$

In this simplified notebook we stay in `[0,1)` space with modulo-1 wrapping.

In [ ]:
from pathlib import Path

import torch
from torch import nn

from kldm.data import CSPTask, resolve_data_root
from kldm.diffusionModels.trivialized_diffusion import TrivialisedDiffusion

torch.manual_seed(7)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
root = resolve_data_root()

diffusion = TrivialisedDiffusion().to(device)
diffusion

## 1. Load A Real Batch

We use `CSPTask` here because it gives us real fractional coordinates `pos` from MP-20 without one-hot atom diffusion.

In [ ]:
batch = next(iter(CSPTask().dataloader(root=root, split='train', batch_size=2, shuffle=False, download=True))).to(device)

print('pos shape:', tuple(batch.pos.shape))
print('first 3 coordinates:')
print(batch.pos[:3])
print('min:', float(batch.pos.min()))
print('max:', float(batch.pos.max()))

## 2. Sample The Forward Kernel

We sample `f_t` and `v_t` using the real class. In this simplified implementation we also return the raw Gaussian noises `epsilon_v` and `epsilon_r`.

In [ ]:
t_graph = torch.rand(batch.num_graphs, device=device)
t_node = t_graph[batch.batch]

f_t, v_t, epsilon_v, epsilon_r = diffusion.forward_sample(t=t_node, f0=batch.pos)

print('t_graph:', t_graph)
print('\nOriginal positions (first 3 atoms):')
print(batch.pos[:3])
print('\nNoisy positions f_t (first 3 atoms):')
print(f_t[:3])
print('\nNoisy velocities v_t (first 3 atoms):')
print(v_t[:3])
print('\nPosition range after wrapping:')
print('min:', float(f_t.min()))
print('max:', float(f_t.max()))

## 3. Velocity Score Target

For the Gaussian velocity kernel we have the exact score target

$$
\nabla_{v_t} \log p(v_t \mid v_0) = -\epsilon_v / \sigma_v(t)
$$

We compute that target below. The wrapped position target is more involved, so in this notebook we keep the focus on the sampled forward variables and the velocity target.

In [ ]:
sigma_v_t = diffusion._match_dims(diffusion.sigma_v(diffusion.time_scaling_T * t_node), v_t)
target_v = -epsilon_v / sigma_v_t

print('target_v shape:', tuple(target_v.shape))
print('first 3 target rows:')
print(target_v[:3])

## 4. A Tiny Score Model For Velocity

As a simple training example, we build a tiny MLP that sees the current position `f_t`, the noisy velocity `v_t`, and time `t`, and predicts the velocity score target.

In [ ]:
class TinyVelocityScoreNet(nn.Module):
    def __init__(self, hidden_dim: int = 128) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3 + 3 + 1, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 3),
        )

    def forward(self, f_t: torch.Tensor, v_t: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        x = torch.cat([f_t, v_t, t[:, None]], dim=-1)
        return self.net(x)


def dsm_loss(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    return ((pred - target) ** 2).mean()


model = TinyVelocityScoreNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
model

## 5. One Training Step

This mirrors the structure of the continuous diffusion notebook, but for the simplified trivialised velocity target.

In [ ]:
batch = next(iter(CSPTask().dataloader(root=root, split='train', batch_size=8, shuffle=True, download=True))).to(device)
t_graph = torch.rand(batch.num_graphs, device=device)
t_node = t_graph[batch.batch]

f_t, v_t, epsilon_v, epsilon_r = diffusion.forward_sample(t=t_node, f0=batch.pos)
sigma_v_t = diffusion._match_dims(diffusion.sigma_v(diffusion.time_scaling_T * t_node), v_t)
target_v = -epsilon_v / sigma_v_t

pred_v = model(f_t, v_t, t_node)
loss = dsm_loss(pred_v, target_v)

optimizer.zero_grad()
loss.backward()
optimizer.step()

print('f_t shape   :', tuple(f_t.shape))
print('v_t shape   :', tuple(v_t.shape))
print('target shape:', tuple(target_v.shape))
print('pred shape  :', tuple(pred_v.shape))
print('loss        :', float(loss))

## 6. Small Training Loop

Finally, here is a tiny training loop over real CSP batches.

In [ ]:
train_loader = CSPTask().dataloader(root=root, split='train', batch_size=16, shuffle=True, download=True)
num_epochs = 5
print_every = 20

for epoch in range(num_epochs):
    running_loss = 0.0

    for step, batch in enumerate(train_loader):
        batch = batch.to(device)
        t_graph = torch.rand(batch.num_graphs, device=device)
        t_node = t_graph[batch.batch]

        f_t, v_t, epsilon_v, epsilon_r = diffusion.forward_sample(t=t_node, f0=batch.pos)
        sigma_v_t = diffusion._match_dims(diffusion.sigma_v(diffusion.time_scaling_T * t_node), v_t)
        target_v = -epsilon_v / sigma_v_t

        pred_v = model(f_t, v_t, t_node)
        loss = dsm_loss(pred_v, target_v)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += float(loss)

        if step % print_every == 0:
            print(f'epoch={epoch:02d} step={step:03d} loss={float(loss):.6f}')

    mean_loss = running_loss / (step + 1)
    print(f'epoch={epoch:02d} mean_loss={mean_loss:.6f}')